# Eksplorasi Fitur TF-IDF & Model ML Klasik (SVM, Naive Bayes, Random Forest)
**Proyek:** Analisis Sentimen Reputasi Undiksha pada Media Sosial  
**Tahap:** 5B – Fitur TF-IDF & Model Klasik (PRD §6 F5, F8)  
**Kepatuhan:** Anti-Leakage (Split stratified sebelum fit, SMOTE hanya pada train set).


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve()))
from src.utils import load_config, set_seed
from src.features.tfidf_vectorizer import prepare_features
from src.models.classical import train_and_evaluate_all_classical

config = load_config('../configs/config.yaml')
set_seed(config['seed'])

df = pd.read_csv('../data/labeled/youtube_labeled.csv')
print(f"Data dimuat: {len(df)} baris")


## 1. Ekstraksi Fitur TF-IDF & Anti-Leakage Split

In [ ]:
bundle = prepare_features(df, config, use_smote=True)
print(f"Bentuk X_train (setelah SMOTE): {bundle['X_train'].shape}")
print(f"Bentuk X_val                 : {bundle['X_val'].shape}")
print(f"Bentuk X_test                : {bundle['X_test'].shape}")


## 2. Pelatihan & Evaluasi Model ML Klasik

In [ ]:
results = train_and_evaluate_all_classical(bundle, config)

summary_rows = []
for model_name, res in results.items():
    tm = res['test_metrics']
    summary_rows.append({
        'Model': model_name.upper(),
        'Accuracy': tm['accuracy'],
        'F1-Macro': tm['f1_macro'],
        'Precision': tm['precision_macro'],
        'Recall': tm['recall_macro'],
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))


## 3. Visualisasi Perbandingan Model Klasik

In [ ]:
plt.figure(figsize=(8, 4))
x = np.arange(len(summary_df))
width = 0.35

plt.bar(x - width/2, summary_df['Accuracy'], width, label='Accuracy', color='#3498db')
plt.bar(x + width/2, summary_df['F1-Macro'], width, label='F1-Macro', color='#2ecc71')

plt.xticks(x, summary_df['Model'])
plt.ylim(0, 1.0)
plt.ylabel('Skor')
plt.title('Perbandingan Kinerja Model ML Klasik (Test Set)')
plt.legend()
plt.tight_layout()
plt.show()
